# Ch 4: GPT Architecture
- The GPT Model has the following components as shown in the figure below:
    1. Embedding layers
    2. Dropout layer
    3. Transformer layer
        - Multi-head causal attention layer
        - Feed Forwarding layer
        - Shorcut paths
    4. LayerNorm layer
    5. Output layer

<p align="center"><img src="Images/Screenshot 2025-06-29 220054.png" width="700" height=""></p>

- In the previous section we have discussed about the importance of embedding,dropout and attention layers.
- Now, we will discuss one by one about the importance of each layer which we have not discussed till now. 

### Layer Normalization Layer
- This layer is basically needed for feature scaling using z-score normalization, squishing all the data into the same range (mean = 0, std = 1), for better training.
- But it still uses 2 trainable parameters: 
    1. Scale($\gamma$)
    2. Shift($\beta$)
- We find layer normalization using the following formula:
$$LayerNorm(x)=\gamma.\frac{x-\mu}{\sigma}+\beta$$

- These parameters allow the model to scale values in it's desired range, desired range not necessarily between squished range in z-score .
- Even though the normalised data is suitable for training faster but, sometimes the differences between the values are unnoticable, but still these differences need to be noticed.

**Example:** 
- Sometimes both the sentences seems similar but has different sence

- **Sentence 1:** "Wow, what a great job you did."
- **Sentence 2:** "Wow, what a great job you did!"

- The tone in the first one (due to punctuation and context) might be sarcastic, while the second is likely sincere.
- If we just normalise the values of theses sentences, we will get almost the same number,which means there is no difference between these 2 sentences.
- But for an LLM, it is necessary to understand the sense of the sentence. Hence, it must allow the values to scale and shift a bit for these differences to be noticed or this differences to be noticed

**Conclusion**
- The scaling (γ) and shifting (β) parameters in LayerNorm allow the model to re-expand important differences that were normalized away.

**NOTE:**
- During normalisation we add the parameter `eps` to variance to prevent division by zero error.
- If we set `unbiased=False` while calculating variance using `var()`, it means we using the formula $\frac{\sum_i (x_i-\bar{x})^2}{n}$. This formula doesn't include the bessels correction i.e, by dividing the denominator by `n-1`.

**Algorithm**
1. In `__init__`, declare eps,scale and shift
2. Find mean and variance
3. Find z-score normalization(eps must be added to std. deviation)
4. Multiply scale and add shift,then return

In [1]:
import torch
import torch.nn as nn
class LayerNorm(nn.Module):
    def __init__(self,dim):
        # Declare eps,scale and shift
        super().__init__()
        self.eps=1e-5
        self.scale=nn.Parameter(torch.ones(dim))
        self.shift=nn.Parameter(torch.zeros(dim))
    def forward(self,x):
        # Find mean and variance
        mean=x.mean(dim=-1, keepdim=True)
        var=x.var(dim=-1, keepdim=True, unbiased=False)
        # Find z-score normalization
        # eps added to variance to prevent divided by zero error
        x=(x-mean)/torch.sqrt(var+self.eps)
        # Multiply scale and add shift,then return
        return self.scale*x+self.shift

### Transformer Layer
- The Transformer layer is the core building block of the GPT model, and it is crucial for enabling the model to understand and generate language effectively. 

- Let's understand the components of Transformer layer given in the diagram below

<p align="center"><img src="Images/Screenshot 2025-06-30 110531.png" width="" height=""></p>

- It uses 2 key layers:
    1. **Multi Head Causal Attention layers:** Captures the relationship between words and returns context vector, which gives the probable meanings of this sentence captured by each attention head. 
    2. **Feed Forwarding Layer:** It emphasizes more relevant meanings and suppressing less important ones by passing through activation function.
- Other than these 2 keys layers , it uses:
    1. LayerNorm(for feature scaling)
    2. Dropout(to reduce overfitting)
    3. Shortcut Paths(to prevent vanishing gradients)
- In most transformer based architechtures like GPTs,BERT, multiple layers of transformer blocks are stacked on each other.
- Number of layers of transformer block used tells about the level of thinking of the transformer architechture.
- Just like if some of our friends are smart we say, he/she can think much level higher than us.
- Just look at some capabilities of a GPT model with different number of transformer blocks.We can see the model becomes more smart as the number of transformer blocks increases.

| **Layer Depth** | **Analogy**              | **What It Enables GPT To Do**                                     |
|-----------------|--------------------------|--------------------------------------------------------------------|
| 1–2 layers      | Word spotting            | Basic word relationships, no deep understanding                   |
| 4–6 layers      | Sentence level           | Understands sentence structure and phrasing                       |
| 8–12 layers     | Paragraph level          | Tracks multiple sentences, resolves references                    |
| 24+ layers      | Document-level reasoning | Tracks logic, multi-step reasoning, world knowledge               |
| 48+ layers      | Abstract reasoning       | Handles instructions, complex prompts, creativity                 |

- Lets look at the number of transformer blocks used by different versions of GPT model.

| **Model Variant** | **Number of Transformer Layers** | **Capabilities Summary**                                |
|-------------------|-----------------------------------|----------------------------------------------------------|
| GPT-2 Small        | 12                                | Simple sentence-level generation                         |
| GPT-2 Medium       | 24                                | More fluent, better context tracking                     |
| GPT-2 Large        | 36                                | Paragraph-level coherence                                |
| GPT-3              | 96                                | Advanced reasoning, few-shot learning                    |
| GPT-4 (est.)       | 96–128+                           | Instruction-following, abstraction, creativity           |

- We will discuss each key component of transformer layer.

### Feed Forwarding Layer
- In the previous chapter we read about the Multihead Causal Attention, in which each head captures multiple meanings of the sentence with probalities.
- The most relevant(most probable) meaning should be emphasised to reduce ambiguity in a sentence.
- This task is performed by **Feed Forwarding layer**.
- It captures the most relevant meaning by:
    1. Increasing dimention size to 4 times to understand the context vector in a deeper level.
    2. Suppressing the negetive weights to a value near to zero using GELU activation function.

<p align="center"><img src="Images/Screenshot 2025-06-30 120436.png" width="300" height=""></p>

**Drawbacks of ReLU**
- In GPT-1 we were using ReLU as activation function.
- Even though ReLU is easy for computation it lead to an issue of dead neurons.
-  Dead neurons are those neurons which always outputs zero and don't participate in learning.
- The negative values in ReLU activation are turned to zero, and they donot participate in training.
- By chance,in a neuron if the value becomes less than zero, that, neuron losses it's training ability and gradient remains zero for any number of epochs.
- To deal this new activation functions like LeakyReLU,PReLU,GELU,SwiGLU are introduced.

### GELU(Gaussian Error Linear Unit) 
- This function is very similar to ReLU function, but it has smooth curve at zero,unlike pointed curve in ReLU as shown in the diagram below.
- Slope at negetive values is not equal to zero(except at minima -0.75).
- Hence, it make neurons with negative values trainable.

<p align="center"><img src="Images/Screenshot 2025-06-30 143107.png" width="" height=""></p>

- This activation function is proposed in paper [Hendrycks and Gimpel 2016](https://arxiv.org/abs/1606.08415).

- The eqution for this activation function is:
$$GELU(x)\approx 0.5.x.(1+tanh[\sqrt{\frac{2}{\pi}}.(x+0.044715.x^3)])$$ 

In [2]:
class GELU(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, x):
        return 0.5 * x * (1 + torch.tanh(
            torch.sqrt(torch.tensor(2.0 / torch.pi))*(x + 0.044715 * torch.pow(x, 3))
        ))

**Algorithm(for Feed Forwarding Layer)**
1. In `__init__` create feed forwarding layer with:
    - First layer is linear layer expanding dimention to 4 times
    - Second layer is GELU activation function
    - Third layer is linear returns the original dimention size
2. Pass the input through layer


In [3]:
class FeedForward(nn.Module):
    def __init__(self,dim):
        super().__init__()
        self.layers=nn.Sequential(
            nn.Linear(dim,4*dim),# Expanding Layer
            GELU(),# Activation Function
            nn.Linear(4*dim,dim)# Contracting Layer
        )
    def forward(self,x):
        return self.layers(x)

### Shorcut Connections
- Shorcut connections represents the paths which allows the values to pass without passing through layers.
- It is introduced to mitigate the problem of vanishing gradients created by activation functions.
- In GELU function, gradients becomes very small as they more negetive valued nodes, since negetive values have very less gradient compared to positive values.
- Shortcuts prevents vanishing gradients in the following ways:
    1. The shorcut paths provide a way to pass gradients through them ensuring non-zero gradients throughout the training.
    2. If the output of a layer are mostly near to zeroes, addition of input through shorcut layer restores the positive negetive diversity.
- Lets understand this with the help of example given below:


In [4]:
class ExampleDeepNeuralNetwork(nn.Module):
    def __init__(self, layer_sizes, use_shortcut):
        super().__init__()
        self.use_shortcut = use_shortcut
        self.layers = nn.ModuleList([
            nn.Sequential(nn.Linear(layer_sizes[0], layer_sizes[1]), GELU()),
            nn.Sequential(nn.Linear(layer_sizes[1], layer_sizes[2]), GELU()),
            nn.Sequential(nn.Linear(layer_sizes[2], layer_sizes[3]), GELU()),
            nn.Sequential(nn.Linear(layer_sizes[3], layer_sizes[4]), GELU()),
            nn.Sequential(nn.Linear(layer_sizes[4], layer_sizes[5]), GELU())
        ])

    def forward(self, x):
        for layer in self.layers:
            # Compute the output of the current layer
            layer_output = layer(x)
            # Check if shortcut can be applied
            if self.use_shortcut and x.shape == layer_output.shape:
                x = x + layer_output
            else:
                x = layer_output
        return x


def print_gradients(model, x):
    # Forward pass
    output = model(x)
    target = torch.tensor([[0.]])

    # Calculate loss based on how close the target
    # and output are
    loss = nn.MSELoss()
    loss = loss(output, target)
    
    # Backward pass to calculate the gradients
    loss.backward()

    for name, param in model.named_parameters():
        if 'weight' in name:
            # Print the mean absolute gradient of the weights
            print(f"{name} has gradient mean of {param.grad.abs().mean().item()}")

- Lets check the gradients without shortcut connections.

In [5]:
layer_sizes = [3, 3, 3, 3, 3, 1]  

sample_input = torch.tensor([[1., 0., -1.]])

torch.manual_seed(123)
model_without_shortcut = ExampleDeepNeuralNetwork(
    layer_sizes, use_shortcut=False
)
print_gradients(model_without_shortcut, sample_input)

layers.0.0.weight has gradient mean of 0.00020173587836325169
layers.1.0.weight has gradient mean of 0.0001201116101583466
layers.2.0.weight has gradient mean of 0.0007152041653171182
layers.3.0.weight has gradient mean of 0.001398873864673078
layers.4.0.weight has gradient mean of 0.005049646366387606


- Lets check the gradients with shortcut connections.

In [6]:
torch.manual_seed(123)
model_with_shortcut = ExampleDeepNeuralNetwork(
    layer_sizes, use_shortcut=True
)
print_gradients(model_with_shortcut, sample_input)

layers.0.0.weight has gradient mean of 0.22169792652130127
layers.1.0.weight has gradient mean of 0.20694106817245483
layers.2.0.weight has gradient mean of 0.32896995544433594
layers.3.0.weight has gradient mean of 0.2665732502937317
layers.4.0.weight has gradient mean of 1.3258541822433472


-  We can see that gradients are much higher if we allow shorcut connections.

<p align="center"><img src="Images/Screenshot 2025-06-30 182733.png" width="" height=""></p>

### GPT Configuration
- The following things to the mentioned in a GPT Configuration:
1. **Vocabulary Size(`vocab_size`):** Total number of unique tokens. GPT2 has a vocabulary size of 50257
2. **Context length:(`context_length`)** Number of tokens in feature and target label
3. **Dimention(`dim`):** Dimention size of embedding
4. **Number of Heads in attention(`num_heads`)**
5. **Number of transformer layers(`num_layers`)**
6. **Dropout rate(`drop_rate`)**
7. **Query Key Value bias(`qkv_bias`)**

- Lets look at the configuration of different types of GPT-2 models which you can try with:

|**GPT Model**|**dim**|**num_layers**|**num_heads**|
|-|-|-|-|
|GPT2-small|768|12|12|
|GPT2-medium|1024|24|16|
|GPT2-large|1280|36|20|
|GPT2-XL|1600|48|25|

In [7]:
GPT2_CONFIG = {
    "vocab_size": 50257,    # Vocabulary size
    "context_length": 1024, # Context length
    "dim": 768,         # Embedding dimension
    "num_heads": 12,          # Number of attention heads
    "num_layers": 12,         # Number of layers
    "drop_rate": 0.1,       # Dropout rate
    "qkv_bias": False       # Query-Key-Value bias
}

### Building Tranformer Layer
- In `__init__`
    1. Declare Attention,Feed forward,2 Layer norm layers and Dropout layer
- In `forward()`
    1. Passing through Attention layer
    2. Passing through Feed Feedforward layer

In [8]:
from GPTModules import MultiheadAttention

class TransformerBlock(nn.Module):
    def __init__(self,cfg):
        super().__init__()
        # Declare Attention,FeedForward,LayerNorm and Dropout layers
        self.attn=MultiheadAttention(
            dim=cfg["dim"],
            dropout=cfg["drop_rate"],
            num_heads=cfg["num_heads"],
            context_length=cfg["context_length"],
            qkv_bias=cfg["qkv_bias"]
        )
        self.ff=FeedForward(cfg["dim"])
        # Separate LayerNorm layers are used for each
        self.norm1=LayerNorm(cfg["dim"])
        self.norm2=LayerNorm(cfg["dim"])
        self.dropout=nn.Dropout(cfg["drop_rate"])
    def forward(self,x):
        # Passing through Attention layer
        shortcut=x
        x=self.norm1(x)
        x=self.attn(x)
        x=self.dropout(x)
        x=x+shortcut
        # Passing through Feed Feedforward layer
        shortcut=x
        x=self.norm2(x)
        x=self.ff(x)
        x=self.dropout(x)
        x=x+shortcut

        return x

- **NOTE:** We need to use separate layer norm layers for attention and feed forwarding, since both of them are different and model may require different scaling and shifting for them
- Let's check if our transformer block is working properly.

In [9]:
torch.manual_seed(123)

x = torch.rand(2, 4, 768)  # Shape: [batch_size, num_tokens, emb_dim]
block = TransformerBlock(GPT2_CONFIG)
output = block(x)

print("Input shape:", x.shape)
print("Output shape:", output.shape)

Input shape: torch.Size([2, 4, 768])
Output shape: torch.Size([2, 4, 768])


### Building GPT Model
- In `__init__`
    1. Declare token and positional embeddings
    2. Declare dropout,transformer blocks, layer norm and output head
- In `forward`
    1. Find shape of the input batch
    2. Find postional and token embeddings and add them
    3. Pass through dropoout,transformer blocks,layer norm output head

In [10]:
class GPTModel(nn.Module):
    def __init__(self,cfg):
        super().__init__()
        self.tok_emb=nn.Embedding(cfg["vocab_size"],cfg["dim"])
        self.pos_emb=nn.Embedding(cfg["context_length"],cfg["dim"])
        self.dropout=nn.Dropout(cfg["drop_rate"])
        self.trf_blocks=nn.Sequential(
            *[TransformerBlock(cfg) for _ in range(cfg["num_layers"])]
        )
        self.layer_norm=LayerNorm(cfg["dim"])
        self.out_head=nn.Linear(
            cfg["dim"],cfg["vocab_size"],bias=False
        )
    def forward(self,input):
        batch_size,context_len=input.shape
        tok_emb=self.tok_emb(input)
        pos_emb=self.pos_emb(torch.arange(context_len,device=input.device))
        x=tok_emb+pos_emb
        x=self.dropout(x)
        x=self.trf_blocks(x)
        x=self.layer_norm(x)
        x=self.out_head(x)
        return x

In [11]:
import tiktoken

tokenizer = tiktoken.get_encoding("gpt2")

batch = []

txt1 = "Every effort moves you"
txt2 = "Every day holds a"

batch.append(torch.tensor(tokenizer.encode(txt1)))
batch.append(torch.tensor(tokenizer.encode(txt2)))
batch = torch.stack(batch, dim=0)
print(batch)

tensor([[6109, 3626, 6100,  345],
        [6109, 1110, 6622,  257]])


In [12]:
torch.manual_seed(123)

model = GPTModel(GPT2_CONFIG)
batch=batch.to("cuda")
model.to("cuda")
out = model(batch)
print("Input batch:\n", batch)
print("\nOutput shape:", out.shape)

Input batch:
 tensor([[6109, 3626, 6100,  345],
        [6109, 1110, 6622,  257]], device='cuda:0')

Output shape: torch.Size([2, 4, 50257])


## Generating Text
- We follow the following procedure for the same:
    1. Pass the input tokens to the GPTModel
    2. Extract the last token of the output given by GPTModel
    3. Append the last token to the input token and follow step 1 untill desired number of new tokens are added.


In [ ]:
def generate_text_ids(model,input,new_tokens,window_size):
    for _ in range(new_tokens):
        input_cropped=input[:,-window_size:]
        with torch.no_grad():
            logits=model(input_cropped)
        logits=logits[:,-1,:]
        probas=torch.softmax(logits,dim=-1)
        next_token_id=torch.argmax(probas,dim=-1,keepdim=True)
        input=torch.cat((input,next_token_id),dim=-1)
    return input
        


In [16]:
sentence="This is a"
tokenised_sentence=tokenizer.encode(sentence)
input_batch=torch.tensor([tokenised_sentence])
model.eval()
model.to("cpu")
out=generate_text_ids(
    model=model,
    input=input_batch,
    new_tokens=6,
    window_size=GPT2_CONFIG["context_length"]
)
decoded_out=tokenizer.decode(out.squeeze(0).tolist())
print(decoded_out)

This is a Curse ker Contains vivohen lit


In [16]:
import torch
import torch.nn as nn
x=torch.tensor(
    [
        [1,2,3],
        [4,5,6]
    ]
)
x=x.flatten()
print(x)

tensor([1, 2, 3, 4, 5, 6])
